<a href="https://colab.research.google.com/github/caroarcrz/Bin-packing-problem-Contenedor-Refrigerado/blob/main/Contenedor_refigerado1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimizando el llenado de un contenedor refrigerado

El problema de la optimización en el llenado de un contenedor refrigerado consiste en determinar el acomodo de un
conjunto de cajas de frutas y verduras dentro de un contenedor utilizado para transporte terrestre. El contenedor cuenta
con una unidad de enfriamiento ubicada en la parte frontal, cercana a la cabina del conductor, por lo que la temperatura
no se distribuye de manera uniforme. En general, las zonas cercanas al sistema de refrigeración presentan temperaturas
más bajas, mientras que las zonas cercanas a la puerta trasera tienden a presentar temperaturas más elevadas

Para representar este comportamiento térmico de forma sencilla, el contenedor se divide en un conjunto de zonas
térmicas. Cada zona térmica contiene un número determinado de hileras, cada hilera permite acomodar un número
máximo de cajas a lo ancho, y cada posición puede tener varios niveles de apilamiento. De esta manera, el contenedor
se representa como un conjunto de posiciones discretas donde pueden colocarse cajas.

Cada tipo de producto tiene una cantidad fija de cajas que debe ser transportada, así como características propias
de peso, dimensiones, resistencia al apilamiento y sensibilidad térmica. Los productos más sensibles a la temperatura
deben colocarse preferentemente en las zonas más frías del contenedor, mientras que los productos menos sensibles
pueden ubicarse en zonas de mayor temperatura. El objetivo del problema es asignar las cajas de cada producto a las
posiciones disponibles dentro del contenedor, minimizando el deterioro térmico total y respetando las restricciones
físicas de capacidad, apilamiento y factibilidad de carga.

In [77]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#PRIMERO SE INSTALARAN LAS PAQUETERIAS QUE SE UTILIZARAN PARA EL MODELADO

In [78]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 47.2 MB/s eta 0:00:00


In [79]:
import gurobipy as gp
from gurobipy import GRB

In [82]:
# ============================================================
# CONJUNTOS
# ============================================================

In [84]:
P = ['F','P','J']          # Fresa, Papa, Jitomate
Z = [1,2,3]                # Zonas
H = [1]                    # Una hilera
A = [1,2]                  # Dos posiciones
L = [1,2,3]                # Tres niveles

print('Los conjuntos son:')
print('P =', P)
print('Z =', Z)
print('H =', H)
print('A =', A)
print('L =', L)

Los conjuntos son:
P = ['F', 'P', 'J']
Z = [1, 2, 3]
H = [1]
A = [1, 2]
L = [1, 2, 3]


In [85]:
# ============================================================
# PARAMETROS
# ============================================================

In [86]:
# Número de cajas n_p
n = {
    'F':5,
    'P':7,
    'J':3
}

# Peso de cada caja (kg) w_p
w = {
    'F':4,
    'P':7,
    'J':5
}

# Peso máximo que resiste una caja s_p
s = {
    'F':10,
    'P':20,
    'J':15
}

# Coeficiente de deterioro  alpha_p
alpha = {
    'F':10,
    'P':2,
    'J':8
}

# Temperatura por zona. theta_z
theta = {
    1:2,
    2:8,
    3:10
}

print('El número de cajas son', n)
print('El peso de cada caja es', w)
print('El peso máximo que resiste una caja es', s)
print('El coeficiente de deterioro es', alpha)
print('La temperatura por zona es', theta)

El número de cajas son {'F': 5, 'P': 7, 'J': 3}
El peso de cada caja es {'F': 4, 'P': 7, 'J': 5}
El peso máximo que resiste una caja es {'F': 10, 'P': 20, 'J': 15}
El coeficiente de deterioro es {'F': 10, 'P': 2, 'J': 8}
La temperatura por zona es {1: 2, 2: 8, 3: 10}


In [87]:
# ============================================================
# PENALIZACION
# ============================================================

In [88]:
d = {}

for p in P:
    for z in Z:
        d[p,z] = alpha[p]*theta[z]

In [89]:
m = gp.Model("Acomodo")

# ============================================================
# VARIABLES BINARIAS
# x[p,z,h,a,l]
# ============================================================

x = m.addVars(P,Z,H,A,L,
              vtype=GRB.BINARY,
              name="x")

# ============================================================
# FUNCIÓN OBJETIVO
# ============================================================

m.setObjective(

    gp.quicksum(
        d[p,z]*x[p,z,h,a,l]
        for p in P
        for z in Z
        for h in H
        for a in A
        for l in L
    ),

    GRB.MINIMIZE

)

# ============================================================
# RESTRICCIÓN (2)
# Número de cajas de cada producto
# ============================================================

for p in P:

    m.addConstr(

        gp.quicksum(
            x[p,z,h,a,l]
            for z in Z
            for h in H
            for a in A
            for l in L
        ) == n[p],

        name=f"Cantidad_{p}"

    )

# ============================================================
# RESTRICCIÓN (3)
# Una caja por posición
# ============================================================

for z in Z:
    for h in H:
        for a in A:
            for l in L:

                m.addConstr(

                    gp.quicksum(
                        x[p,z,h,a,l]
                        for p in P
                    ) <= 1,

                    name=f"Ocupacion_{z}_{h}_{a}_{l}"

                )

# ============================================================
# RESTRICCIÓN (4)
# No puede haber cajas flotando
# ============================================================

for z in Z:
    for h in H:
        for a in A:
            for l in [2,3]:

                m.addConstr(

                    gp.quicksum(
                        x[p,z,h,a,l]
                        for p in P
                    )

                    <=

                    gp.quicksum(
                        x[p,z,h,a,l-1]
                        for p in P
                    ),

                    name=f"Apilamiento_{z}_{a}_{l}"

                )

# ============================================================
# RESTRICCIÓN (5)
# Peso máximo soportado
# ============================================================

for z in Z:
    for h in H:
        for a in A:
            for l in [1,2]:

                peso_superior = gp.quicksum(

                    w[p]*x[p,z,h,a,ll]

                    for p in P
                    for ll in range(l+1,4)

                )

                resistencia = gp.quicksum(

                    s[p]*x[p,z,h,a,l]

                    for p in P

                )

                m.addConstr(

                    peso_superior <= resistencia,

                    name=f"Peso_{z}_{a}_{l}"

                )

# ============================================================
# OPTIMIZAR
# ============================================================

m.optimize()

# ============================================================
# RESULTADOS
# ============================================================

if m.status == GRB.OPTIMAL:

    print("\n")
    print("="*60)
    print("VALOR ÓPTIMO")
    print("="*60)
    print(m.objVal)

    print("\n")

    for z in Z:

        print("="*40)
        print(f"ZONA {z}")
        print("="*40)

        for a in A:

            print(f"\nPosición {a}")

            for l in [3,2,1]:

                producto = "-"

                for p in P:

                    if x[p,z,1,a,l].X > 0.5:
                        producto = p

                print(f"Nivel {l}: {producto}")

else:

    print("No se encontró solución óptima.")

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 45 rows, 54 columns and 270 nonzeros (Min)
Model fingerprint: 0xb3d95ce2
Model has 54 linear objective coefficients
Variable types: 0 continuous, 54 integer (54 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [4e+00, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+00]

Found heuristic solution: objective 624.0000000
Presolve removed 6 rows and 0 columns
Presolve time: 0.00s
Presolved: 39 rows, 54 columns, 234 nonzeros
Variable types: 0 continuous, 54 integer (54 binary)

Root relaxation: objective 3.680000e+02, 24 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |   